# Robot Sensor Fusion with EKF and UKF

Academic team project — sanitised portfolio edition.


## Introduction  

This project aims to apply multidimensional Kalman filters for sensor data fusion in nonlinear systems. The **Extended Kalman Filter (EKF)** and the **Unscented Kalman Filter (UKF)** will be explored to estimate the position of a robot.  

The robot is modeled as a nonlinear dynamic system and is equipped with the following sensors:  
- **Speed sensor**  
- **Gyroscope**  
- **GNSS (Global Navigation Satellite System) sensor**  

The simulation will allow a comparison between the trajectories estimated by the EKF and UKF against the robot’s actual trajectory, assessing the accuracy and robustness of each method.  

## Project Structure  

The development will be carried out in Python and will follow these steps:  

1. **Implement the robot's dynamic model**.  
2. **Construct the observation model**.  
3. **Simulate sensor data acquisition**.  
4. **Develop the Extended Kalman Filter (EKF)**.  
5. **Develop the Unscented Kalman Filter (UKF)**.  
6. **Configure the parameters** of the robot and filters.  
7. **Simulate the robot's movement** and visualize the estimated trajectories.  
8. **Evaluate filter performance** using metrics such as **MAE, RMSE and R²**.  
9. **Analyze and interpret** the obtained results. 



**Libraries used** 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib.animation as animation
from IPython.display import display, clear_output
import scipy

### **Parameters Configuration (Point 6)**

In [ ]:
# Time step (Δt)
dt = 0.1  # seconds

# Process noise covariance matrix (Q): represents uncertainty in the dynamic model
Q = np.diag([
    0.1,  # variance of location on x-axis
    0.1,  # variance of location on y-axis
    np.deg2rad(1.0),  # variance of yaw angle
    1.0  # variance of velocity
]) ** 2  # predict state covariance


# Measurement noise covariance matrix (R): represents GNSS sensor uncertainty in x and y
R = np.diag([1.0, 1.0]) ** 2  # Observation x,y position covariance

# Initial state vector (x0): [x, y, phi, v]
# Example: starting at the origin, facing along 0 radians, with a velocity of 0 m/s.
x0 = np.array([0.0, 0.0, 0.0, 0.0])

# Initial state covariance matrix (P0): uncertainty in the initial state
P0 = np.eye(4)

# Control input vector (u): [v, phi_rate]
u_true = np.array([1.0, 0.1]) # v = 1 m/s, phi_rate = 0.1 rad/s

# Observation matrix H, which extracts the x and y positions from the state.
H = np.array([[1, 0, 0, 0],
              [0, 1, 0, 0]])

# Sensor noise standard deviations:
# - For the speed sensor and gyroscope (input vector u = [v_measured, phi_rate])
input_noise_std = np.array([0.5, 0.5])**2
#input_noise_std = np.array([1.7, 0.25])

# - For the GNSS sensor (observation vector z = [x, y])
#observation_noise_std = np.array([1.0, np.deg2rad(30.0)])**2 
observation_noise_std = np.array([0.2, 0.2]) 

# UKF scaling parameters:
ALPHA = 1e-3
BETA = 2
KAPPA = 0

### **Dynamic Model of the robot implementation (Point 1)**

The `robot_dynamic_model` function updates the state of a **robot** using a **discrete-time motion model**. It takes the current state `x`, control input `u`, and time step `dt`, then computes the next state `x_next` using a **linear state transition matrix (`A`)** and a **control input matrix (`B`)**.

- **State Vector (`x`)**: `[x, y, phi, v]`  
  - `x, y`: 2D position  
  - `phi`: Orientation (heading angle)  
  - `v`: Velocity  

- **Control Input (`u`)**: `[v_measured, phi_rate]`  
  - `v_measured`: Measured speed  
  - `phi_rate`: Gyroscope-measured turn rate  

- **Matrices Used**:  
  - `A`: Keeps positions and orientation unchanged (except velocity).  
  - `B`: Uses `cos(phi)` and `sin(phi)` to transform velocity into position changes.  

**Function Behavior**  
1. Extracts the **orientation** (`phi`) from `x`.  
2. Constructs **B**, which determines how control inputs (`u`) affect `x`.  
3. Computes the **next state** using the formula: x_next = A @ x + B @ u


In [ ]:
def robot_dynamic_model(x, u, dt):
    A = np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0],
                  [0, 0, 1, 0],
                  [0, 0, 0, 0]])
    
    # Extracts the current orientation (phi) from the state vector.
    phi = x[2]
    
    B = np.array([[np.cos(phi) * dt, 0],
                  [np.sin(phi) * dt, 0],
                  [0, dt],
                  [1, 0]])
    
    x_next = A @ x + B @ u
    return x_next

### **Robot observation model implementation (Point 2)**

The `observation_model` function extracts the **x-y position** from the robot and its **state vector** using an **observation matrix (`H`)**.

- **State Vector (`x`)**: `[x, y, phi, v]`  
  - `x, y`: 2D position  
  - `phi`: Orientation (heading angle)  
  - `v`: Velocity  

- **Observation Matrix (`H`)**:  
  - Extracts only the **position components (`x, y`)**  
  - Defined as:  
  $$
    H =  
    \begin{bmatrix}
    1 & 0 & 0 & 0 \\
    0 & 1 & 0 & 0
    \end{bmatrix}
  $$
    

**Function Behavior**  
1. Takes the **state vector** and **observation matrix** as inputs.  
2. Computes the **observation vector (`z`)** using matrix multiplication: z = H @ x 
3. Returns `z`, which contains **only the robot’s x-y position**.

In [ ]:
def observation_model(x, H):
    z = H @ x 
    return z

### **Simulate sensor data acquisition (Point 3)**

These functions **simulate sensor data acquisition** for an **Extended Kalman Filter (EKF)** and an **Unscented Kalman Filter (UKF)** by:
1. **Updating the true state** of the robot using a noise-free motion model.
2. **Generating a noisy GPS observation** from the state.
3. **Adding noise to the control input** for dead-reckoning.
4. **Updating the dead-reckoning state** using the noisy input.

---
**Function: `acquire_sensor_data_ekf`**
- Uses a **direct noisy measurement approach** for GPS observation.
- **Noise Model:**
  - GPS noise added directly to `x_true` values.
  - Input noise applied **independently** to `u`.

**Steps:**
1. **True State Update**  
   - Computes the next state using the **robot’s motion model** (noise-free).
  
2. **Generate Noisy GPS Observation**  
   - Adds **Gaussian noise** to the **x, y** components of `x_true`.
  
3. **Apply Noise to Control Input**  
   - Adds **Gaussian noise** to the input vector `u`.
  
4. **Update Dead-Reckoning State**  
   - Uses the **noisy input** to update `x_dead`.

---

**Function: `acquire_sensor_data_ukf`**
- Uses a **matrix-based noise injection approach** for UKF.
- **Noise Model:**
  - GPS noise applied using **matrix multiplication** (`observation_noise_std @ random_noise`).
  - Input noise is added using the **same matrix-based approach**.

**Steps:**
1. **True State Update**  
   - Computes the next state using the **robot’s motion model** (noise-free).
  
2. **Generate Noisy GPS Observation**  
   - Uses the **observation model** (`H * x`) and **adds noise** via matrix multiplication.
  
3. **Apply Noise to Control Input**  
   - Uses a **matrix-based noise application** for `u_noisy`.
  
4. **Update Dead-Reckoning State**  
   - Uses the **noisy input** to update `x_dead`.

---

Both functions serve the same **core purpose** but use different noise-handling techniques tailored to their respective filtering methods (EKF vs. UKF).

In [ ]:
def acquire_sensor_data_ekf(x, u, x_dead, input_noise_std, observation_noise_std):
    # True state update (noise-free)
    x_true = robot_dynamic_model(x, u, dt=dt)

    # Creates a noisy GPS observation
    zx = x_true[0] + np.random.randn() * observation_noise_std[0]
    zy = x_true[1] + np.random.randn() * observation_noise_std[1]
    z_noisy = np.array([zx, zy]) 

    # Adds noise to the input
    ud1 = u[0] + np.random.randn() * input_noise_std[0]
    ud2 = u[1] + np.random.randn() * input_noise_std[1]
    u_noisy = np.array([ud1, ud2]) 
    
    # Dead-reckoning update (integrate noisy input)
    x_dead = robot_dynamic_model(x_dead, u_noisy, dt=dt)

    return x_true, z_noisy, x_dead, u_noisy

def acquire_sensor_data_ukf(x, u, x_dead, input_noise_std, observation_noise_std):

    # True state update (noise-free)
    x_true = robot_dynamic_model(x, u, dt=dt)

    # Creates a noisy GPS observation
    z_noisy = observation_model(x_true, H) + observation_noise_std @ np.random.randn(2, 1)

    # Adds noise to the input
    u_noisy = u + input_noise_std @ np.random.randn(2, 1)

    # Dead-reckoning update (integrate noisy input)
    x_dead = robot_dynamic_model(x_dead, u_noisy, dt=dt)

    return x_true, z_noisy, x_dead, u_noisy

### **Extended Kalman Filter (EKF) development (Point 4)**

These functions implement the **Extended Kalman Filter (EKF)** for state estimation of a robot. The EKF is used to estimate the state of a nonlinear system by:
1. **Predicting** the next state based on a motion model.
2. **Updating** the state estimate using sensor measurements.

---

**Function: `ekf_state_transition(x, u, dt)`**
- **Predicts** the next state using a nonlinear motion model.
- Uses the current state `[x, y, phi, v]` and control input `[v_measured, phi_rate]` to compute the **next state**.
- The velocity `v` influences the **x, y position updates**.
- The orientation `phi` is updated based on `phi_rate`.

---

**Function: `compute_jacobian(x, dt)`**
- **Computes the Jacobian matrix (F)** of the state transition function.
- The Jacobian linearizes the nonlinear motion model around the current state to improve EKF accuracy.
- Used in error covariance propagation.

**Jacobian Matrix (F)**
$$
F =
\begin{bmatrix}
1 & 0 & -v \sin(\phi) dt & \cos(\phi) dt \\
0 & 1 & v \cos(\phi) dt & \sin(\phi) dt \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

---

**Function: `ekf_update(x_prev, P_prev, u, z, dt, Q, R)`**
- **Performs the EKF update step**, which includes:
  1. **Predicting the next state** using `ekf_state_transition`.
  2. **Propagating the error covariance** using the Jacobian.
  3. **Computing the innovation (measurement residual)**.
  4. **Updating the state estimate** using the Kalman Gain.

---

In [ ]:
def ekf_state_transition(x, u, dt):
    # Extracts current orientation and velocity from the state vector.
    phi = x[2]
    v = x[3]
    
    # Computes the predicted state based on the dynamic model.
    x_pred = np.array([
        x[0] + np.cos(phi) * dt * v,   # Update x position
        x[1] + np.sin(phi) * dt * v,   # Update y position
        x[2] + dt * u[1],              # Update orientation (phi)
        u[0]                         # Set velocity to the measured value
    ])
    return x_pred

def compute_jacobian(x, dt):
    # Orientation and velocity from the current state.
    phi = x[2]
    v = x[3]
    
    F = np.array([
        [1, 0, -np.sin(phi) * dt * v, np.cos(phi) * dt],
        [0, 1,  np.cos(phi) * dt * v, np.sin(phi) * dt],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])

    return F

def ekf_update(x_prev, P_prev, u, z, dt, Q, R):
    # Propagates the state using the state transition function, given the previous state, control input, and time step.
    x_pred = ekf_state_transition(x_prev, u, dt)
    
    # Computes the Jacobian matrix of the state transition function with respect to the state.
    F = compute_jacobian(x_prev, dt)
    
    # Propagates the error covariance. F @ P_prev @ F.T applies the linearized transformation
    #    to the previous covariance, and Q adds the process noise.
    P_pred = F @ P_prev @ F.T + Q

    # Uses the observation model to predict the measurement from the predicted state.
    z_pred = observation_model(x_pred, H)

    # Calculates the innovation (measurement residual): the difference between the actual measurement and the prediction.
    y_innov = z - z_pred
    
    # Computes the innovation covariance S which incorporates the uncertainty from both the predicted state and the measurement noise.
    S = H @ P_pred @ H.T + R
    
    # Computes the Kalman gain K that balances the trust between the prediction and the new measurement.
    K = P_pred @ H.T @ np.linalg.inv(S)

    # Updates the state estimate by applying the Kalman gain to the innovation.
    x_upd = x_pred + K @ y_innov
    
    # Updates the error covariance matrix. This effectively "shrinks" the predicted covariance based on how much the measurement improved our state estimate.
    P_upd = (np.eye(4) - K @ H) @ P_pred

    # Returns the updated state estimate and error covariance matrix.
    return x_upd, P_upd

### **Unscented Kalman Filter (UKF) development (Point 5)**

**Sigma Point Generation**
`generate_sigma_points(xEst, PEst, gamma)`
- Computes sigma points around the estimated state for the Unscented Kalman Filter (UKF).  
- Uses **matrix square root decomposition** (`scipy.linalg.sqrtm`) instead of Cholesky decomposition for numerical stability.  
- Returns a set of `2*nx + 1` sigma points.  
- Ensures **efficient computation** by extracting individual columns of the square root matrix.  

**Motion Model Prediction**
`predict_sigma_motion(sigma, u)`
- Propagates each sigma point through the motion model using control input `u`.
- Applies the `robot_dynamic_model()` function to predict the next state.

**Measurement Model Prediction**
`predict_sigma_observation(sigma)`
- Transforms sigma points into measurement space using `observation_model()`.
- Produces predicted measurement sigma points.

**Covariance Computation**
`calc_sigma_covariance(x, sigma, wc, Pi)`
- Computes the covariance matrix of the sigma points.
- Uses weighted deviations from the mean and adds noise covariance.

**Cross-Covariance Calculation**
 `calc_pxz(sigma, x, z_sigma, zb, wc)`
- Computes the cross-covariance between state and measurement sigma points.
- Essential for calculating the Kalman gain.

**UKF Update Step**
`ukf_update(xEst, PEst, z, u, wm, wc, gamma)`
- Performs the full UKF cycle, including:
  - Generating sigma points.
  - Predicting state and covariance.
  - Updating with new measurements.
  - Computing the Kalman gain.
- Returns updated state estimate and covariance.

**UKF Setup**
`setup_ukf(nx)`
- Computes weights for sigma points and the scaling factor `gamma`.
- Uses UKF parameters `ALPHA`, `BETA`, and `KAPPA` to determine `lambda`.


In [ ]:
def generate_sigma_points(xEst, PEst, gamma):
    nx = xEst.shape[0]  # Number of state variables

    # Initializes sigma points array
    sigma = np.zeros((2 * nx + 1, nx))

    # First sigma point is the mean state
    sigma[0] = xEst

    # Computes square root of covariance matrix once
    sqrt_P = scipy.linalg.sqrtm(PEst)

    # Generates sigma points
    for i in range(nx):
        sigma[i + 1]      = xEst + gamma * sqrt_P[:, i]  # Positive direction
        sigma[nx + i + 1] = xEst - gamma * sqrt_P[:, i]  # Negative direction

    return sigma


def predict_sigma_motion(sigma, u):
    # Number of sigma points
    nsigma = sigma.shape[0]

    # Initializes the predicted sigma points array with the same shape.
    sigma_pred = np.zeros_like(sigma)
    
    # For each sigma point, apply the motion model.
    # This propagates the sigma point through the robot_dynamic_model.
    for i in range(nsigma):
        sigma_pred[i] = robot_dynamic_model(sigma[i], u, dt)
    
    return sigma_pred


def predict_sigma_observation(sigma):
    # Number of sigma points
    nsigma = sigma.shape[0]

    # Initializes an empty list to collect predicted measurement for each sigma point.
    z_sigma = []
    
    # For each sigma point, computes the observation using the observation_model.
    for i in range(nsigma):
        z_i = observation_model(sigma[i], H)
        z_sigma.append(z_i)
    
    # Converts the list to a NumPy array.
    return np.array(z_sigma)


def calc_sigma_covariance(x, sigma, wc, Pi):
    # Starts with the provided initial covariance.
    P = Pi.copy()
    
    # Compute the weighted sum of outer products
    for i in range((sigma.shape[0])):  # Iterate over sigma points
        diff = sigma[i] - x
        P += wc[i] * np.outer(diff, diff.T)
    
    return P


def calc_pxz(sigma, x, z_sigma, zb, wc):
    # Gets the dimensions of the state and measurement vectors.
    nx = x.shape[0]
    z_dim = z_sigma.shape[1]

    # Initializes the cross-covariance matrix.
    Pxz = np.zeros((nx, z_dim))
    
    # Sums over the weighted outer products of the differences.
    for i in range(len(sigma)):
        dx = sigma[i] - x         # deviation of state sigma point from state mean
        dz = z_sigma[i] - zb      # deviation of measurement sigma point from measurement mean
        Pxz += wc[i] * np.outer(dx, dz.T)
    
    return Pxz


def ukf_update(xEst, PEst, z, u, wm, wc, gamma):
    # ------------------ Prediction (Time Update) ------------------
    # Generates sigma points based on the current state and covariance.
    sigma = generate_sigma_points(xEst, PEst, gamma)
    
    # Propagates each sigma point through the motion model.
    sigma_pred = predict_sigma_motion(sigma, u)
    
    # Computes the predicted state mean using the weighted sum of the sigma points.
    xPred = np.sum(wm[:, None] * sigma_pred, axis=0)
    
    # Computes the predicted state covariance by combining process noise Q.
    PPred = calc_sigma_covariance(xPred, sigma_pred, wc, Q)



    # ------------------ Update (Measurement Update) ------------------
    # Predicts the measurement from the predicted state using the observation model.
    zPred = observation_model(xPred, H)
    
    # Computes the innovation (measurement residual) between actual and predicted measurement.
    y = z - zPred

    # Generates new sigma points from the predicted state and covariance.
    sigma2 = generate_sigma_points(xPred, PPred, gamma)
    
    # Propagates these sigma points through the observation model.
    z_sigma = predict_sigma_observation(sigma2)
    
    # Computes the predicted measurement mean (zb) as the weighted sum of z_sigma.
    zb = np.sum(wm[:, None] * z_sigma, axis=0)
    
    # Computes the measurement covariance (st) including measurement noise R.
    st = calc_sigma_covariance(zb, z_sigma, wc, R)
    
    # Computes the cross-covariance between the state and measurement sigma points.
    Pxz = calc_pxz(sigma2, xPred, z_sigma, zb, wc)
    
    # Calculates the Kalman gain using the cross-covariance and measurement covariance.
    Kt = Pxz @ np.linalg.inv(st)
    
    # Updates the state estimate with the Kalman gain and measurement residual.
    xEst = xPred + Kt @ (z - zb)
    
    # Updates the state covariance estimate.
    PEst = PPred - Kt @ st @ Kt.T

    return xEst, PEst


def setup_ukf(nx):
    # Calculates lambda using the UKF scaling parameters.
    lamb = ALPHA**2 * (nx + KAPPA) - nx
    
    # The first weight for the mean is computed as lambda divided by (lambda + nx).
    wm = [lamb / (lamb + nx)]
    # The first weight for the covariance has an additional term (1 - ALPHA**2 + BETA).
    wc = [lamb / (lamb + nx) + (1 - ALPHA**2 + BETA)]
    
    # For the remaining sigma points, assign equal weights.
    for _ in range(2 * nx):
        wm.append(1.0 / (2.0 * (nx + lamb)))
        wc.append(1.0 / (2.0 * (nx + lamb)))
    
    # Calculates the scaling factor gamma.
    gamma = math.sqrt(nx + lamb)
    
    return np.array(wm), np.array(wc), gamma

### **Filter performance evaluation using performance indicators (MAE, RMSE and R²) (Point 8)**

The `compute_errors` function evaluates the accuracy of an estimated trajectory compared to the true trajectory using three key error metrics: RMSE, MAE, and R-squared.

**Parameters**
- `true_traj` (numpy array): The ground truth trajectory.
- `est_traj` (numpy array): The estimated trajectory.

**Returns**
- `rmse` (numpy array): Root Mean Square Error (RMSE) for each state dimension.
- `mae` (numpy array): Mean Absolute Error (MAE) for each state dimension.
- `r_squared` (numpy array): R-squared (coefficient of determination) indicating the goodness of fit.

**Error Metrics**
1. **Root Mean Square Error (RMSE)**
   - Measures the standard deviation of the estimation error.
   - Lower values indicate better accuracy.

2. **Mean Absolute Error (MAE)**
   - Computes the average absolute difference between the estimated and true values.
   - Less sensitive to large errors compared to RMSE.

3. **R-squared (R²)**
   - Evaluates how well the estimated trajectory explains the variance in the true trajectory.
   - A value close to 1 indicates a good fit, while values near 0 suggest poor estimation.

In [ ]:
def compute_errors(true_traj, est_traj):
    # Computes the errors between the true and estimated trajectories.
    errors = true_traj - est_traj
    squared_errors = errors ** 2

    # RMSE
    rmse = np.sqrt(np.mean(squared_errors, axis=0))

    # MAE
    mae = np.mean(np.abs(errors), axis=0)

    # R-squared (Coefficient of Determination)
    ss_total = np.sum((true_traj - np.mean(true_traj, axis=0)) ** 2, axis=0)
    ss_residual = np.sum(squared_errors, axis=0)
    r_squared = 1 - (ss_residual / ss_total)
    
    return rmse, mae, r_squared

### **Main function + Robot's movement simulation (Point 7)**

**Key Features**
- **Choice of Filter:** The variable `ekf_or_not` determines whether the simulation uses EKF (`1`) or UKF (`0`).
- **Trajectory Initialization:** Initializes the true state, dead reckoning state, estimated state, and covariance matrix.
- **Visualization:** Uses Matplotlib to display trajectories:
  - **Blue Line:** True trajectory
  - **Black Line:** Dead-reckoning trajectory
  - **Red Line:** Estimated trajectory (EKF/UKF)
  - **Green Dots:** Observations
- **Real-Time Updates:** The plot is updated at each step for a live visualization of state estimation performance.

**Main Steps**
1. **Setup and Initialization**
   - Defines simulation parameters.
   - Initializes states and filter settings.
   - Creates a Matplotlib figure for visualization.

2. **Simulation Loop (`for step in range(num_steps)`):**
   - **Sensor Data Acquisition:** Uses either EKF or UKF sensor models to get noisy measurements.
   - **State Estimation:** Updates state estimates using EKF or UKF update functions.
   - **Logging:** Stores true, estimated, dead-reckoning, and observation trajectories.
   - **Visualization Update:** Plots the real-time trajectory changes.

3. **Final Analysis**
   - Computes **Root Mean Square Error (RMSE)**, **Mean Absolute Error (MAE)**, and **R-squared (R²)** for evaluating filter accuracy.
   - Displays the results for EKF or UKF.

In [ ]:
def plot_covariance_ellipse(xEst, PEst):
    Pxy = PEst[0:2, 0:2]  # Extract the covariance matrix for x and y
    eigval, eigvec = np.linalg.eig(Pxy)

    if eigval[0] >= eigval[1]:
        bigind = 0
        smallind = 1
    else:
        bigind = 1
        smallind = 0

    t = np.linspace(0, 2 * np.pi, 100)
    a = np.sqrt(eigval[bigind])  # Semi-major axis
    b = np.sqrt(eigval[smallind])  # Semi-minor axis
    x = a * np.cos(t)
    y = b * np.sin(t)

    # Compute rotation angle
    angle = np.arctan2(eigvec[bigind, 1], eigvec[bigind, 0])
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle), np.cos(angle)]])
    
    # Rotate and translate the ellipse
    xy_ellipse = R @ np.array([x, y])  # Ensure correct matrix multiplication
    px = xy_ellipse[0, :] + xEst[0]  # Ensure xEst is indexed correctly
    py = xy_ellipse[1, :] + xEst[1]

    plt.plot(px, py, "--r")  # Plot the covariance ellipse



if __name__ == "__main__":
    num_steps = 500  # Total number of simulation steps

    x_true = x0.copy()  # True state
    x_dead = x0.copy()  # Dead reckoning state
    
    # Separate state and covariance for EKF and UKF
    x_est_ekf = x0.copy()
    P_est_ekf = P0.copy()
    x_est_ukf = x0.copy()
    P_est_ukf = P0.copy()

    # Store trajectories
    true_traj = [x_true[:2].copy()]
    dead_traj = [x_dead[:2].copy()]
    est_traj_ekf = [x_est_ekf[:2].copy()]
    est_traj_ukf = [x_est_ukf[:2].copy()]
    obs_traj = []

    wm, wc, gamma = setup_ukf(x0.shape[0])  # Setup UKF parameters

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(-20, 20)
    ax.set_ylim(-1, 25)
    ax.set_xlabel("X position (m)")
    ax.set_ylabel("Y position (m)")
    ax.grid(True)
    ax.axis("equal")

    for step in range(num_steps):
        # Acquire sensor data
        x_true, z_noisy, x_dead, u_noisy = acquire_sensor_data_ekf(
            x_true, u_true, x_dead, input_noise_std, observation_noise_std
        )
        
        # EKF Update
        x_est_ekf, P_est_ekf = ekf_update(x_est_ekf, P_est_ekf, u_noisy, z_noisy, dt, Q, R)
        
        # UKF Update
        x_est_ukf, P_est_ukf = ukf_update(x_est_ukf, P_est_ukf, z_noisy, u_noisy, wm, wc, gamma)

        # Log states
        true_traj.append(x_true[:2].copy())
        dead_traj.append(x_dead[:2].copy())
        est_traj_ekf.append(x_est_ekf[:2].copy())
        est_traj_ukf.append(x_est_ukf[:2].copy())
        obs_traj.append(z_noisy.copy())

        # Convert lists to numpy arrays for plotting
        true_traj_array = np.array(true_traj)
        dead_traj_array = np.array(dead_traj)
        est_traj_ekf_array = np.array(est_traj_ekf)
        est_traj_ukf_array = np.array(est_traj_ukf)
        obs_traj_array = np.array(obs_traj)

        # Clear previous frame
        plt.cla()

        # Re-draw plot settings
        plt.xlim(-20, 20)
        plt.ylim(-1, 25)
        plt.xlabel("X position (m)")
        plt.ylabel("Y position (m)")
        plt.grid(True)
        plt.axis("equal")

        # Re-plot updated trajectories
        plt.plot(true_traj_array[:, 0], true_traj_array[:, 1], "b-", label="True Trajectory")
        plt.plot(dead_traj_array[:, 0], dead_traj_array[:, 1], "k-", label="Dead-Reckoning")
        plt.plot(est_traj_ekf_array[:, 0], est_traj_ekf_array[:, 1], "r-", label="EKF Estimate")
        plt.plot(est_traj_ukf_array[:, 0], est_traj_ukf_array[:, 1], "m-", label="UKF Estimate")
        plt.plot(obs_traj_array[:, 0], obs_traj_array[:, 1], "go", label="Observations", markersize=2)
        plot_covariance_ellipse(x_est_ekf, P_est_ekf)
        plot_covariance_ellipse(x_est_ukf, P_est_ukf)
        
        # Re-add legend
        plt.legend(loc="upper right")

        # Display updated plot
        clear_output(wait=True)
        display(fig)

    # Close the plot window after simulation
    plt.close()

    # Compute and print error metrics
    rmse_ekf, mae_ekf, r2_ekf = compute_errors(true_traj_array, est_traj_ekf_array)
    rmse_ukf, mae_ukf, r2_ukf = compute_errors(true_traj_array, est_traj_ukf_array)

    print("===== EKF Performance =====")
    print(f"RMSE: {rmse_ekf}")
    print(f"MAE: {mae_ekf}")
    print(f"R-squared: {r2_ekf}")
    
    print("\n===== UKF Performance =====")
    print(f"RMSE: {rmse_ukf}")
    print(f"MAE: {mae_ukf}")
    print(f"R-squared: {r2_ukf}")

# **Final Conclusions and Comments (Point 9)**  

For this comparison, 10 outputs from each filter were analyzed.  

## **Accuracy Comparison (RMSE, MAE, R²)**  

### **Root Mean Square Error (RMSE)**  
**EKF RMSE (X, Y):**  
- **Best:** [0.0869, 0.0880]  
- **Worst:** [0.1417, 0.1413]  
- **Average:** [0.1113, 0.1102]  

**UKF RMSE (X, Y):**  
- **Best:** [0.0717, 0.0671]  
- **Worst:** [0.1319, 0.1233]  
- **Average:** [0.1043, 0.0987]  

**Observations:**  
- UKF had lower RMSE values in most cases, indicating better accuracy in tracking the true trajectory.  
- The best-case RMSE of UKF was **better** than EKF’s best-case RMSE.  
- The worst-case RMSE of EKF was **higher**, suggesting it had more deviations in some cases.  

---

### **Mean Absolute Error (MAE)**  
**EKF MAE (X, Y):**  
- **Best:** [0.0712, 0.0702]  
- **Worst:** [0.1125, 0.1100]  
- **Average:** [0.0902, 0.0891]  

**UKF MAE (X, Y):**  
- **Best:** [0.0569, 0.0518]  
- **Worst:** [0.1053, 0.0936]  
- **Average:** [0.0841, 0.0784]  

**Observations:**  
- UKF consistently had **lower MAE values** than EKF, meaning its predictions were closer to the actual values on average.  
- The difference in MAE is small but noticeable, favoring **UKF as the more reliable estimator**.  

---

### **R-squared Value (R²)**  
**EKF R² (X, Y):**  
- **Best:** [0.99985, 0.99983]  
- **Worst:** [0.99960, 0.99954]  
- **Average:** [0.99973, 0.99971]  

**UKF R² (X, Y):**  
- **Best:** [0.99990, 0.99989]  
- **Worst:** [0.99966, 0.99965]  
- **Average:** [0.99979, 0.99977]  

**Observations:**  
- **UKF had slightly better R² values**, meaning it explained more variance in the true state.  
- The difference is marginal, indicating that both filters performed well.  

---

## **Trajectory Analysis**  

Examining the plotted **true trajectory, dead reckoning, estimated trajectory, and observations**:  

- **The estimated trajectory (red) closely followed the true trajectory (blue)**, with only minimal deviations.  
- **Dead reckoning (black) showed more noticeable deviations**, confirming the need for filtering.  
- **Observations (green) contained noise**, but both EKF and UKF effectively handled it.  
- **UKF’s estimated trajectory appeared smoother**, suggesting it managed noise slightly better.  
- **Both filters tracked the true trajectory well**, with only minor deviations within expected limits.  

---
